In [0]:
import os
import requests
from urllib.parse import urlparse

In [0]:
def get_token():
    auth_endpoint = f"{API_ENDPOINT}/authentication/v1/token/access-key"
    payload = {
        "accessKeyId": dbutils.secrets.get(scope="nice_credentials", key="accessKey"),
        "accessKeySecret": dbutils.secrets.get(scope="nice_credentials", key="accessSecret")
    }
    try:
        response = requests.post(auth_endpoint, json=payload)
        response.raise_for_status()
        data = response.json()
        return data["access_token"]
    except requests.exceptions.HTTPError as e:
        print(f"[HTTP Error] get_token: {e.response.status_code} - {e.response.reason}")
    return None

def get_contacts(token, start_date, end_date, max_results=10):
    url = f"{API_ENDPOINT}/incontactapi/services/v34.0/contacts/completed"
    headers = {
        "Authorization": f"Bearer {token}"
    }
    params = {
        "startDate": start_date,
        "endDate": end_date,
        "top": max_results,
        "orderBy": "lastUpdateTime DESC",
    }
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as e:
        print(f"[HTTP Error] get_contacts: {e.response.status_code} - {e.response.reason}")
    return None

def get_audio_recording(token, acd_call_id):
    """Retrieve audio recording metadata (including playback URL) for a given ACD call ID"""
    url = f"{API_ENDPOINT}/media-playback/v1/contacts"
    params = {
        "acd-call-id": acd_call_id,
        # "media-type": "all",
        # "exclude-waveforms": "true",
    }
    headers = {"Authorization": f"Bearer {token}"}
    try:
        response = requests.get(url, params=params, headers=headers)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 404:
            return None
        else:
            return 'Error:' + str(e.response.status_code)
    return None

def download_audio(url, output_path):
    """Download audio recording"""
    response = requests.get(url, allow_redirects=True)
    response.raise_for_status()

    with open(output_path, "wb") as f:
        f.write(response.content)
    #print(f"Audio saved to {output_path}")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

In [0]:
API_ENDPOINT = "https://api-na1.niceincontact.com"
token = get_token()

In [0]:
contacts_data = get_contacts(token, "2026-03-15", "2026-03-31")

In [0]:
for contact in contacts_data["completedContacts"]:
    contact_id = contact["contactId"]
    recording_info = get_audio_recording(token, contact_id)
    #print(contact_id, recording_info)
    if recording_info is None:
        print(f"No recording found for contact ID: {contact_id}")
    # elif "playbackUrl" not in recording_info:
    #     print(f"No playback URL found for contact ID: {contact_id}")
    else:
        print(json.dumps(recording_info, indent=4))
        url = recording_info["interactions"][0]["data"]["fileToPlayUrl"]
        filename = os.path.basename(urlparse(url).path)
        output_path = f"/Workspace/Users/v578213@amerisourcebergen.com/{filename}"
        #download_audio(url, output_path)

In [0]:
import json
print(json.dumps(contacts_data["completedContacts"], indent=4))

   {
        "abandoned": false,
        "abandonSeconds": 0.0,
        "acwSeconds": 20.01,
        "agentId": 44098574,
        "agentSeconds": 30.02,
        "analyticsProcessedDate": null,
        "callbackTime": 0,
        "campaignId": 7833429,
        "campaignName": "MWI VM/OB",
        "conferenceSeconds": 0.0,
        "contactId": 698754403923,
        "contactStartDate": "2026-03-30T21:08:20.063Z",
        "dateACWWarehoused": "2026-03-31T13:24:45.920Z",
        "dateContactWarehoused": "2026-03-31T13:24:47.653Z",
        "dispositionNotes": null,
        "endReason": null,
        "firstName": "John",
        "fromAddress": "6602873945",
        "highProficiency": 1,
        "holdCount": 0,
        "holdSeconds": 0.0,
        "inQueueSeconds": 58448.06,
        "isAnalyticsProcessed": false,
        "isLogged": false,
        "isOutbound": false,
        "isRefused": false,
        "isShortAbandon": false,
        "isTakeover": false,
        "lastName": "Bilgri",
        "lastUpdateTime": "2026-03-31T13:24:47.653Z",
        "lowProficiency": 20,
        "masterContactId": 698754402612,
        "mediaSubTypeId": null,
        "mediaSubTypeName": null,
        "mediaTypeId": 5,
        "mediaTypeName": "VoiceMail",
        "pointOfContactId": 8,
        "pointOfContactName": "inContactInboundVoiceMailPOC",
        "postQueueSeconds": 0.0,
        "preQueueSeconds": 0.007,
        "primaryDispositionId": null,
        "refuseReason": null,
        "refuseTime": null,
        "releaseSeconds": 0.006,
        "routingAttribute": -1,
        "routingTime": 11587,
        "secondaryDispositionId": null,
        "serviceLevelFlag": 1,
        "skillId": 23044899,
        "skillName": "MWI IVESCO Swine VM",
        "teamId": 9233917,
        "teamName": "MWI Production Animal West",
        "toAddress": "8003970204",
        "totalDurationSeconds": 58478.0859,
        "transferIndicatorId": 0,
        "transferIndicatorName": "None"
    },

{
    "contactId": "6a6b20b9-595d-40aa-b2aa-c105d975d55d",
    "acdcontactId": "698754691506",
    "elevatedInteraction": false,
    "interactions": [
        {
            "mediaType": "voice-and-screen",
            "channelType": "PHONE_CALL",
            "startTime": "2026-03-31T05:58:40.577Z",
            "endTime": "2026-03-31T06:00:44.11Z",
            "data": {
                "startTime": "2026-03-31T05:58:40.577Z",
                "endTime": "2026-03-31T06:00:44.11Z",
                "acwEndTime": "2026-03-31T06:00:44.11Z",
                "fileToPlayUrl": "https://production-mcrplaybackmanager-fzgrfhdt0tfl.s3.us-west-2.amazonaws.com/xxx_.mp4",
                "videoImageUrl": null,
                "waveformDataList": [],
                "participantDataList": [
                    {
                        "participantType": "AGENT",
                        "agentName": "Jaime Limon",
                        "participantId": "11ec466b-8d8a-2a50-8ae9-0242ac110003",
                        "segmentId": null,
                        "userId": "11ec466b-8d8a-2a50-8ae9-0242ac110003",
                        "voiceStages": [
                            {
                                "stageType": "ACTIVE",
                                "startTime": "2026-03-31T05:58:40.604Z",
                                "endTime": "2026-03-31T06:00:44.084Z",
                                "displays": null,
                                "mergedParentId": null
                            }
                        ],
                        "screenStages": [
                            {
                                "stageType": "ACTIVE",
                                "startTime": "2026-03-31T05:58:40.606Z",
                                "endTime": "2026-03-31T06:00:44.123Z",
                                "displays": [
                                    {
                                        "width": 1920,
                                        "height": 1080,
                                        "topLeftX": 0,
                                        "topLeftY": 0
                                    },
                                    {
                                        "width": 1920,
                                        "height": 1080,
                                        "topLeftX": 1920,
                                        "topLeftY": 0
                                    },
                                    {
                                        "width": 1920,
                                        "height": 1080,
                                        "topLeftX": 3840,
                                        "topLeftY": 0
                                    }
                                ],
                                "mergedParentId": null
                            }
                        ],
                        "channel": 0
                    },
                    {
                        "participantType": "CUSTOMER",
                        "agentName": null,
                        "participantId": "698754691506",
                        "segmentId": null,
                        "userId": null,
                        "voiceStages": [
                            {
                                "stageType": "ACTIVE",
                                "startTime": "2026-03-31T05:58:40.604Z",
                                "endTime": "2026-03-31T06:00:44.084Z",
                                "displays": null,
                                "mergedParentId": null
                            }
                        ],
                        "screenStages": [],
                        "channel": 1
                    }
                ],
                "segmentsDataList": [
                    {
                        "startTime": "2026-03-31T05:58:40.577Z",
                        "endTime": "2026-03-31T06:00:44.11Z",
                        "acwEndTime": "2026-03-31T06:00:44.11Z",
                        "openReasonType": "SEGMENT",
                        "closeReasonType": "SEGMENT",
                        "directionType": "OUT_BOUND",
                        "source": null,
                        "firstSegment": false,
                        "originalSourceFromSdr": "inContactACD",
                        "isCustomerRestricted": false,
                        "segmentId": "79f7a19d-e16e-4c22-8b7d-5a38863c37fe",
                        "channelType": "PHONE_CALL",
                        "divisionNo": null,
                        "customerRestricted": false
                    }
                ],
                "categoryMatchesList": [],
                "sentiments": [],
                "dfoStages": null
            },
            "@type": "call",
            "callTaggingList": [],
            "dynamicBusinessData": []
        }
    ]
}



{
    "contactId": "b8bb3f2b-dd5c-4b92-b467-85edec39bb81",
    "acdcontactId": "698754689408",
    "elevatedInteraction": false,
    "interactions": [
        {
            "mediaType": "voice-and-screen",
            "channelType": "PHONE_CALL",
            "startTime": "2026-03-31T05:47:25.254Z",
            "endTime": "2026-03-31T05:50:06.53Z",
            "data": {
                "startTime": "2026-03-31T05:47:25.254Z",
                "endTime": "2026-03-31T05:50:06.53Z",
                "acwEndTime": "2026-03-31T05:51:08.091Z",
                "fileToPlayUrl": "https://production-mcrplaybackmanager-fzgrfhdt0tfl.s3.us-west-2.amazonaws.com/xxx_.mp4",
                "videoImageUrl": null,
                "waveformDataList": [],
                "participantDataList": [
                    {
                        "participantType": "AGENT",
                        "agentName": "Jaime Limon",
                        "participantId": "11ec466b-8d8a-2a50-8ae9-0242ac110003",
                        "segmentId": null,
                        "userId": "11ec466b-8d8a-2a50-8ae9-0242ac110003",
                        "voiceStages": [
                            {
                                "stageType": "ACTIVE",
                                "startTime": "2026-03-31T05:47:25.286Z",
                                "endTime": "2026-03-31T05:50:06.535Z",
                                "displays": null,
                                "mergedParentId": null
                            }
                        ],
                        "screenStages": [
                            {
                                "stageType": "ACTIVE",
                                "startTime": "2026-03-31T05:47:25.289Z",
                                "endTime": "2026-03-31T05:50:06.537Z",
                                "displays": [
                                    {
                                        "width": 1920,
                                        "height": 1080,
                                        "topLeftX": 0,
                                        "topLeftY": 0
                                    },
                                    {
                                        "width": 1920,
                                        "height": 1080,
                                        "topLeftX": 1920,
                                        "topLeftY": 0
                                    },
                                    {
                                        "width": 1920,
                                        "height": 1080,
                                        "topLeftX": 3840,
                                        "topLeftY": 0
                                    }
                                ],
                                "mergedParentId": null
                            },
                            {
                                "stageType": "WRAP_UP",
                                "startTime": "2026-03-31T05:50:06.537Z",
                                "endTime": "2026-03-31T05:51:08.091Z",
                                "displays": [
                                    {
                                        "width": 1920,
                                        "height": 1080,
                                        "topLeftX": 0,
                                        "topLeftY": 0
                                    },
                                    {
                                        "width": 1920,
                                        "height": 1080,
                                        "topLeftX": 1920,
                                        "topLeftY": 0
                                    },
                                    {
                                        "width": 1920,
                                        "height": 1080,
                                        "topLeftX": 3840,
                                        "topLeftY": 0
                                    }
                                ],
                                "mergedParentId": null
                            }
                        ],
                        "channel": 0
                    },
                    {
                        "participantType": "CUSTOMER",
                        "agentName": null,
                        "participantId": "698754689408",
                        "segmentId": null,
                        "userId": null,
                        "voiceStages": [
                            {
                                "stageType": "ACTIVE",
                                "startTime": "2026-03-31T05:47:25.286Z",
                                "endTime": "2026-03-31T05:50:06.535Z",
                                "displays": null,
                                "mergedParentId": null
                            }
                        ],
                        "screenStages": [],
                        "channel": 1
                    }
                ],
                "segmentsDataList": [
                    {
                        "startTime": "2026-03-31T05:47:25.254Z",
                        "endTime": "2026-03-31T05:50:06.53Z",
                        "acwEndTime": "2026-03-31T05:51:08.091Z",
                        "openReasonType": "SEGMENT",
                        "closeReasonType": "SEGMENT",
                        "directionType": "IN_BOUND",
                        "source": null,
                        "firstSegment": false,
                        "originalSourceFromSdr": "inContactACD",
                        "isCustomerRestricted": false,
                        "segmentId": "f007743a-6b1f-4ccb-9c51-2de1d9c5e23e",
                        "channelType": "PHONE_CALL",
                        "divisionNo": null,
                        "customerRestricted": false
                    }
                ],
                "categoryMatchesList": [],
                "sentiments": [],
                "dfoStages": null
            },
            "@type": "call",
            "callTaggingList": [],
            "dynamicBusinessData": []
        }
    ]
}



{
    "contactId": "08ed3aa4-474d-4c30-a01e-1dd45d34d6ae",
    "acdcontactId": "698754688042",
    "elevatedInteraction": false,
    "interactions": [
        {
            "mediaType": "voice-and-screen",
            "channelType": "PHONE_CALL",
            "startTime": "2026-03-31T05:39:02.859Z",
            "endTime": "2026-03-31T05:42:16.493Z",
            "data": {
                "startTime": "2026-03-31T05:39:02.859Z",
                "endTime": "2026-03-31T05:42:16.493Z",
                "acwEndTime": "2026-03-31T05:42:16.493Z",
                "fileToPlayUrl": "https://production-mcrplaybackmanager-fzgrfhdt0tfl.s3.us-west-2.amazonaws.com/xxx_.mp4",
                "videoImageUrl": null,
                "waveformDataList": [],
                "participantDataList": [
                    {
                        "participantType": "AGENT",
                        "agentName": "Jaime Limon",
                        "participantId": "11ec466b-8d8a-2a50-8ae9-0242ac110003",
                        "segmentId": null,
                        "userId": "11ec466b-8d8a-2a50-8ae9-0242ac110003",
                        "voiceStages": [
                            {
                                "stageType": "ACTIVE",
                                "startTime": "2026-03-31T05:39:02.886Z",
                                "endTime": "2026-03-31T05:42:16.496Z",
                                "displays": null,
                                "mergedParentId": null
                            }
                        ],
                        "screenStages": [
                            {
                                "stageType": "ACTIVE",
                                "startTime": "2026-03-31T05:39:02.887Z",
                                "endTime": "2026-03-31T05:42:16.506Z",
                                "displays": [
                                    {
                                        "width": 1920,
                                        "height": 1080,
                                        "topLeftX": 0,
                                        "topLeftY": 0
                                    },
                                    {
                                        "width": 1920,
                                        "height": 1080,
                                        "topLeftX": 1920,
                                        "topLeftY": 0
                                    },
                                    {
                                        "width": 1920,
                                        "height": 1080,
                                        "topLeftX": 3840,
                                        "topLeftY": 0
                                    }
                                ],
                                "mergedParentId": null
                            }
                        ],
                        "channel": 0
                    },
                    {
                        "participantType": "CUSTOMER",
                        "agentName": null,
                        "participantId": "698754688042",
                        "segmentId": null,
                        "userId": null,
                        "voiceStages": [
                            {
                                "stageType": "ACTIVE",
                                "startTime": "2026-03-31T05:39:02.886Z",
                                "endTime": "2026-03-31T05:42:16.496Z",
                                "displays": null,
                                "mergedParentId": null
                            }
                        ],
                        "screenStages": [],
                        "channel": 1
                    }
                ],
                "segmentsDataList": [
                    {
                        "startTime": "2026-03-31T05:39:02.859Z",
                        "endTime": "2026-03-31T05:42:16.493Z",
                        "acwEndTime": "2026-03-31T05:42:16.493Z",
                        "openReasonType": "SEGMENT",
                        "closeReasonType": "SEGMENT",
                        "directionType": "OUT_BOUND",
                        "source": null,
                        "firstSegment": false,
                        "originalSourceFromSdr": "inContactACD",
                        "isCustomerRestricted": false,
                        "segmentId": "4107b134-f3f8-42b9-8ecf-0bd58a57ce2a",
                        "channelType": "PHONE_CALL",
                        "divisionNo": null,
                        "customerRestricted": false
                    }
                ],
                "categoryMatchesList": [],
                "sentiments": [],
                "dfoStages": null
            },
            "@type": "call",
            "callTaggingList": [],
            "dynamicBusinessData": []
        }
    ]
}